# 01 — BCRD Exchange Rates | Bronze Ingestion (incremental, Auto Loader)

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Ingest the official USD/DOP daily exchange-rate history published by the Central Bank of the Dominican Republic (BCRD) into the Bronze layer, preserving source values and adding ingestion metadata for traceability.

## Data Sources (verified) — two layouts, both kept as published
| Layout (`_source_layout`) | File | Sheet | Coverage | Columns |
|---|---|---|---|---|
| `daily_since_1991` (automatic download) | `TASA_DOLAR_REFERENCIA_MC.xlsx` | `Diaria` | 2 Jan 1991 onward | Año, Mes (`Ene`…`Dic`), Día, Compra, Venta |
| `historico_5y` (manual export, first load) | `BCRD - Histórico diario de tasas.xlsx` | `Histórico` | rolling five years | Año, Mes (number), Día, Compra, Venta + four BCRD appreciation/depreciation reference columns |

Values are stored as **text exactly as read** (month names are not translated here); Silver types them.

## Lakehouse
- **Raw archive:** `/Volumes/rd_economic_intelligence/bronze/bcrd_raw/exchange_rates/<yyyy>/<mm>/` — one folder per arrival month, files never overwritten.
- **File register:** `rd_economic_intelligence.bronze.bcrd_exchange_rates_files` — **Auto Loader** (`binaryFile`, `trigger(availableNow=True)`, shared helper `src/shared/ingestion.py`) records each new workbook once.
- **Target table:** `rd_economic_intelligence.bronze.bcrd_exchange_rates_daily` — one snapshot per archived workbook.
- **Why keep every snapshot:** each BCRD file covers only the last five years. Keeping all files lets the history grow beyond that window; Silver merges them (newest file wins for each date).

## Bronze rules
- Source values are **not** modified, corrected or discarded.
- Column names are converted to Delta-safe snake_case (the source headers contain `|`, `%` and spaces). The mapping is documented in this notebook.
- No business date is built here; `Año/Mes/Día` stay as delivered (text). Date construction and month-name parsing belong to Silver.
- Data-quality checks are **observational** in Bronze: they report, they do not filter.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
# Step 0 — Dependencies: serverless does not ship openpyxl, required by pandas to read .xlsx
dbutils.library.restartPython()

In [0]:
# ============================================================
# Step 1 — Configuration and Auto Loader file discovery
# ============================================================
from datetime import datetime, timezone
import sys, uuid, json, importlib
import pandas as pd
from pyspark.sql import functions as F

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/bcrd_raw/exchange_rates/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/bcrd_raw/_checkpoints/exchange_rates_files/"
FILES_TABLE = f"{CATALOG}.bronze.bcrd_exchange_rates_files"
TARGET_TABLE = f"{CATALOG}.bronze.bcrd_exchange_rates_daily"
LAYOUTS = {"Diaria": "daily_since_1991", "Histórico": "historico_5y"}   # sheet -> layout
REQUIRED = ["Año", "Mes", "Día", "Compra", "Venta"]

# Source header -> Bronze column (Delta-safe). Order matches the source file.
COLUMN_MAPPING = {
    "Año": "anio",
    "Mes": "mes",
    "Día": "dia",
    "Compra": "compra",
    "Compra | Apre/depre acumulada (%)": "compra_apre_depre_acumulada_pct",
    "Compra | Apre/depre interanual (%)": "compra_apre_depre_interanual_pct",
    "Venta": "venta",
    "Venta | Apre/depre acumulada (%)": "venta_apre_depre_acumulada_pct",
    "Venta | Apre/depre interanual (%)": "venta_apre_depre_interanual_pct",
}
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

registered = ING.register_new_files(spark, RAW_PATH, "*.xlsx", CHECKPOINT, FILES_TABLE,
    comment="Bronze: register of BCRD daily exchange-rate workbooks found in the raw archive by Auto Loader (one row per file).")
pending = ING.pending_files(spark, FILES_TABLE, TARGET_TABLE)
print(f"Batch ID: {BATCH_ID} · Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_path'] for f in pending]}")

In [0]:
# ============================================================
# Step 2 — Per pending file: detect layout, structural contract, observational checks, append
# ============================================================
# Bronze rules: values are not modified; checks report, they do not filter.
def as_text(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v).strip()

COLUMNS = list(COLUMN_MAPPING.values())
captured = {}
for f in pending:
    path = ING.local_path(f["_source_path"])
    sheets = pd.ExcelFile(path).sheet_names
    sheet = next((s for s in LAYOUTS if s in sheets), None)
    if sheet is None:
        raise ValueError(f"{f['_source_path']}: no known sheet ({list(LAYOUTS)}) in {sheets}")
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    header_idx = next((i for i in range(15) if [as_text(v) for v in raw.iloc[i, :3]] == REQUIRED[:3]), None)
    if header_idx is None:
        raise ValueError(f"{f['_source_path']}: header {REQUIRED} not found in sheet {sheet}")
    headers = [as_text(v) for v in raw.iloc[header_idx]]
    keep = [i for i, h in enumerate(headers) if h]
    headers = [headers[i] for i in keep]
    unexpected = [h for h in headers if h not in COLUMN_MAPPING]
    missing = [h for h in REQUIRED if h not in headers]
    if missing or unexpected:
        raise ValueError(f"{f['_source_path']}: source layout changed. Missing: {missing} | Unexpected: {unexpected}")
    title = " | ".join(t for t in (as_text(raw.iloc[i, 0]) for i in range(header_idx)) if t)

    rows = []
    for i in range(header_idx + 1, len(raw)):
        vals = {COLUMN_MAPPING[h]: as_text(raw.iat[i, c]) for h, c in zip(headers, keep)}
        if not any(vals.values()):
            continue
        rows.append(tuple(vals.get(c) for c in COLUMNS) + (i + 1,))
    df = spark.createDataFrame(rows, ", ".join(f"{c} string" for c in COLUMNS) + ", _source_row_number int")
    df = ING.lineage(df, f, BATCH_ID, INGESTION_TS) \
        .withColumn("_source_sheet", F.lit(sheet)).withColumn("_source_layout", F.lit(LAYOUTS[sheet])).withColumn("_report_period", F.lit(title))

    checks = {
        "row_count": len(rows),
        "non_numeric_year_or_day": df.filter(~F.col("anio").rlike(r"^\d{4}$") | ~F.col("dia").rlike(r"^\d{1,2}$")).count(),
        "duplicate_year_month_day": df.groupBy("anio", "mes", "dia").count().filter("count > 1").count(),
        "non_positive_rates": df.filter((F.col("compra").cast("double") <= 0) | (F.col("venta").cast("double") <= 0)).count(),
        "buy_greater_than_sell": df.filter(F.col("compra").cast("double") > F.col("venta").cast("double")).count(),
    }
    for name, value in checks.items():
        print(f"  {'INFO' if name == 'row_count' else ('PASS' if value == 0 else 'WARN'):<4} {name}: {value:,}")

    df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)  # one atomic append per file
    loaded = spark.table(TARGET_TABLE).filter((F.col("_batch_id") == BATCH_ID) & (F.col("_source_path") == f["_source_path"])).count()
    assert loaded == checks["row_count"], f"Row count mismatch: read {checks['row_count']}, loaded {loaded}"
    captured[f["_source_path"].split("exchange_rates/")[-1]] = {"layout": LAYOUTS[sheet], "rows": loaded, "dq_checks": checks}

spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Bronze: BCRD official USD/DOP daily buy/sell rates as text, one snapshot per archived workbook (layouts daily_since_1991 and historico_5y), with lineage.'") if spark.catalog.tableExists(TARGET_TABLE) else None

In [0]:
summary = spark.sql(f"""SELECT COUNT(DISTINCT _source_path) AS snapshots, COUNT(*) AS rows,
                        MIN(int(anio)) AS first_year, MAX(int(anio)) AS last_year FROM {TARGET_TABLE}""").first()
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
    "captured": captured, "snapshots_in_bronze": summary["snapshots"], "years": [summary["first_year"], summary["last_year"]]}, ensure_ascii=False))